# Subword Seq2Seq — No-Preprocessing Baseline (trains directly on raw train.csv)

**How to switch modes:** set `EXECUTION_MODE` in the next cell to `"online"` or `"offline"`.

- `"online"` — installs packages, logs in to Hugging Face, tracks the run in MLflow/DagsHub, and pushes the final model/tokenizer to the Hub. Requires Kaggle's internet toggle **on**.
- `"offline"` — skips all of the above, loads the base model from a local Kaggle Dataset input instead of downloading it, and saves the trained model/tokenizer to the working directory instead of pushing to the Hub. Required for the actual competition submission run, since Kaggle disables internet on the scored run.

In [ ]:
EXECUTION_MODE = "offline"  # "online" or "offline" — flip this before a real (scored) submission run
assert EXECUTION_MODE in ("online", "offline")
IS_ONLINE = EXECUTION_MODE == "online"
SEED = 42  # seed study: run the notebook with SEED = 42, 43 and 44
PUBLISH_MODELS = False     # True: upload the trained model to Kaggle Models (needs Internet ON)
KAGGLE_USERNAME = "eeee13"  # owner of the Kaggle Models read by ensemble-pipeline.ipynb
print(f"Running in {EXECUTION_MODE.upper()} mode")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [ ]:
import sys
import subprocess

if IS_ONLINE:
    # Package installs require internet — skipped entirely in offline mode.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dacite>=1.9,<2", "--upgrade"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "transformers", "datasets", "huggingface_hub", "accelerate", "mlflow"], check=True)
    # Force dagshub to skip checking dependencies
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dagshub", "--no-deps"], check=True)
else:
    print("Offline mode: skipping package installation (using the environment's pre-installed packages).")


In [ ]:
from transformers import set_seed 

# Set global random seed for reproducibility
set_seed(SEED)

In [ ]:
if IS_ONLINE:
    import mlflow
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login

    user_secrets = UserSecretsClient()

    # Authorize Hugging Face
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)

    # Pass Dagshub credentials from Kaggle Secrets to Environment Variables
    os.environ["MLFLOW_TRACKING_USERNAME"] = user_secrets.get_secret("MLFLOW_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = user_secrets.get_secret("MLFLOW_PASSWORD")
    MLFLOW_TRACKING_URI = user_secrets.get_secret("MLFLOW_TRACKING_URI")

    # Connect to MLflow server and select the experiment
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment("deep-past-initiative-machine-translation")

    # Check connection with a test run
    with mlflow.start_run(run_name="New Data-Preprocessing Pipeline Test"):
        mlflow.set_tag("author", "author")
        mlflow.log_param("architecture", "mBART / mT5 / Subword Seq2Seq")
        mlflow.log_metric("status", 1.0)
        print("Successful MLflow-Dagshub connection!")
else:
    print("Offline mode: skipping Hugging Face login and MLflow/DagsHub tracking.")


## Shared split, metric and run record (Phase 2)

The next four cells are identical in all four notebooks (copied from `common/` by `tools/sync_shared_cells.py`). They give every model **the same 156 validation documents** (checked by a SHA-256 of the sorted ids), **the same metric** (sacreBLEU-identical BLEU and chrF++ with word n-grams, `tests/test_metrics.py`), write one prediction file plus one JSON run record per model into `results/`, and record the package versions and GPU (`results/env_*.json`).

In [ ]:
# >>> SHARED CELL: common/split.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import hashlib
import random

ID_COL = "oare_id"
VAL_FRACTION = 0.1
SPLIT_SEED = 42
# sha256 of "\n".join(sorted(val_ids)) for the competition train.csv (1561 documents -> 156 val).
EXPECTED_VAL_SHA256 = "1939161cfcb9ff85082f20b36be2362da429f1da14f8a94f289249ffb78e1304"
EXPECTED_N_VAL = 156


def val_ids_sha256(val_ids):
    return hashlib.sha256("\n".join(sorted(map(str, val_ids))).encode("utf-8")).hexdigest()


def make_split(df, id_col=ID_COL, val_fraction=VAL_FRACTION, seed=SPLIT_SEED, check=True):
    """Return (train_df, val_df), split by document id. Rows with empty text are dropped first."""
    df = df.dropna(subset=["transliteration", "translation"]).reset_index(drop=True)
    doc_ids = sorted(df[id_col].astype(str).unique().tolist())
    random.Random(seed).shuffle(doc_ids)
    n_val = max(1, int(len(doc_ids) * val_fraction))
    val_ids = set(doc_ids[:n_val])
    is_val = df[id_col].astype(str).isin(val_ids)
    train_df = df[~is_val].reset_index(drop=True)
    val_df = df[is_val].reset_index(drop=True)
    digest = val_ids_sha256(val_ids)
    print(f"[shared split] train={len(train_df)} rows, val={len(val_df)} rows "
          f"({n_val} documents), val_ids_sha256={digest[:16]}...")
    if check:
        assert n_val == EXPECTED_N_VAL and digest == EXPECTED_VAL_SHA256, (
            "Validation split differs from the shared reference split "
            f"(n_val={n_val}, sha256={digest}). Is train.csv the original competition file?")
    assert not set(train_df[id_col].astype(str)) & val_ids, "document leakage between train and val"
    return train_df, val_df

In [ ]:
# >>> SHARED CELL: common/metrics.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import math
import re
from collections import Counter

# ---------------------------------------------------------------- BLEU (13a)
_TOK_13A_RULES = [
    (re.compile(r'([\{-\~\[-\` -\&\(-\+\:-\@\/])'), r' \1 '),
    (re.compile(r'([^0-9])([\.,])'), r'\1 \2 '),
    (re.compile(r'([\.,])([^0-9])'), r' \1 \2'),
    (re.compile(r'([0-9])(-)'), r'\1 \2 '),
]


def _tokenize_13a(line):
    line = line.replace('<skipped>', '').replace('-\n', '').replace('\n', ' ')
    if '&' in line:
        line = (line.replace('&quot;', '"').replace('&amp;', '&')
                    .replace('&lt;', '<').replace('&gt;', '>'))
    line = f' {line} '
    for pattern, repl in _TOK_13A_RULES:
        line = pattern.sub(repl, line)
    return ' '.join(line.split())


def _word_ngrams(tokens, max_order):
    return Counter(tuple(tokens[i:i + n])
                   for n in range(1, max_order + 1)
                   for i in range(len(tokens) - n + 1))


def bleu_stats(hyp, ref, max_order=4):
    """Sufficient statistics of one segment: [hyp_len, ref_len, correct_1..4, total_1..4]."""
    h = _tokenize_13a(hyp.rstrip()).split()
    r = _tokenize_13a(ref.rstrip()).split()
    h_ng, r_ng = _word_ngrams(h, max_order), _word_ngrams(r, max_order)
    correct, total = [0] * max_order, [0] * max_order
    for ng, c in h_ng.items():
        total[len(ng) - 1] += c
        if ng in r_ng:
            correct[len(ng) - 1] += min(c, r_ng[ng])
    return [len(h), len(r)] + correct + total


def bleu_from_stats(stats, max_order=4):
    sys_len, ref_len = stats[0], stats[1]
    correct, total = stats[2:2 + max_order], stats[2 + max_order:]
    bp = 1.0
    if sys_len < ref_len:
        bp = math.exp(1 - ref_len / sys_len) if sys_len > 0 else 0.0
    if not any(correct):
        return 0.0
    precisions = [0.0] * max_order
    smooth = 1.0
    for n in range(max_order):
        if total[n] == 0:
            break
        if correct[n] == 0:
            smooth *= 2
            precisions[n] = 100.0 / (smooth * total[n])
        else:
            precisions[n] = 100.0 * correct[n] / total[n]
    log_sum = sum(-9999999999 if p == 0 else math.log(p) for p in precisions)
    return bp * math.exp(log_sum / max_order)


# ---------------------------------------------------------------- chrF++
_PUNCTS = set('!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~')


def _chrf_words(sent):
    out = []
    for w in sent.split():
        if len(w) == 1:
            out.append(w)
        elif w[-1] in _PUNCTS:
            out += [w[:-1], w[-1]]
        elif w[0] in _PUNCTS:
            out += [w[0], w[1:]]
        else:
            out.append(w)
    return out


def _chrf_ngrams(sent, char_order, word_order):
    chars = ''.join(sent.split())
    grams = [Counter(chars[i:i + n] for i in range(len(chars) - n + 1))
             for n in range(1, char_order + 1)]
    words = _chrf_words(sent)
    grams += [Counter(' '.join(words[i:i + n]) for i in range(len(words) - n + 1))
              for n in range(1, word_order + 1)]
    return grams


def chrf_stats(hyp, ref, char_order=6, word_order=2):
    """Flattened [hyp, ref, match] counts for every char then word n-gram order."""
    stats = []
    for h, r in zip(_chrf_ngrams(hyp, char_order, word_order),
                    _chrf_ngrams(ref, char_order, word_order)):
        match = sum(min(c, r[g]) for g, c in h.items() if g in r)
        stats += [sum(h.values()) if r else 0, sum(r.values()), match]
    return stats


def chrf_from_stats(stats, beta=2):
    factor = beta ** 2
    avg_p = avg_r = 0.0
    eff = 0
    for i in range(len(stats) // 3):
        n_hyp, n_ref, n_match = stats[3 * i:3 * i + 3]
        if n_hyp > 0 and n_ref > 0:
            avg_p += n_match / n_hyp
            avg_r += n_match / n_ref
            eff += 1
    if eff == 0:
        return 0.0
    avg_p, avg_r = avg_p / eff, avg_r / eff
    if avg_p + avg_r == 0:
        return 0.0
    return 100.0 * (1 + factor) * avg_p * avg_r / (factor * avg_p + avg_r)


# ---------------------------------------------------------------- public API
def _sum_stats(rows):
    return [sum(col) for col in zip(*rows)] if rows else None


def segment_stats(preds, refs):
    """Per-segment (bleu_stats, chrf_stats); lets bootstrap resample without recomputing n-grams."""
    preds = ['' if p is None else str(p) for p in preds]
    refs = ['' if r is None else str(r) for r in refs]
    assert len(preds) == len(refs), f"{len(preds)} predictions vs {len(refs)} references"
    return [bleu_stats(p, r) for p, r in zip(preds, refs)], [chrf_stats(p, r) for p, r in zip(preds, refs)]


def scores_from_segment_stats(bleu_rows, chrf_rows):
    bleu = bleu_from_stats(_sum_stats(bleu_rows))
    chrfpp = chrf_from_stats(_sum_stats(chrf_rows))
    return {"bleu": bleu, "chrfpp": chrfpp, "geo_mean": math.sqrt(max(bleu, 0.0) * max(chrfpp, 0.0))}


def corpus_scores(preds, refs):
    """Corpus BLEU, chrF++ and their geometric mean (the competition score), all on 0-100."""
    return scores_from_segment_stats(*segment_stats(preds, refs))


def sentence_chrfpp(hyp, ref):
    """Sentence-level chrF++, used as the MBR utility."""
    return chrf_from_stats(chrf_stats(hyp, ref))


METRIC_SIGNATURE = "BLEU=sacrebleu-2.x-13a-exp | chrF++=sacrebleu-2.x-c6-w2-b2 | geo_mean=sqrt(BLEU*chrF++)"

In [ ]:
# >>> SHARED CELL: common/runlog.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import datetime
import json
import os

import pandas as pd


RESULTS_DIR = "results"


def finalize_run(system, seed, ids, refs, preds, hyperparameters, log_history=None,
                 out_dir=RESULTS_DIR, notes=None):
    os.makedirs(out_dir, exist_ok=True)
    ids = [str(i) for i in ids]
    refs = ["" if r is None else str(r) for r in refs]
    preds = ["" if p is None else str(p) for p in preds]
    scores = corpus_scores(preds, refs)
    tag = f"{system}__seed{seed}"
    pred_path = os.path.join(out_dir, f"preds_val_{tag}.csv")
    pd.DataFrame({"id": ids, "target_text": refs, "pred_translation": preds}).to_csv(pred_path, index=False)
    record = {
        "system": system,
        "seed": seed,
        "created_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"),
        "metric": METRIC_SIGNATURE,
        "n_val_docs": len(ids),
        "val_ids_sha256": val_ids_sha256(ids),
        "final_val": scores,
        "hyperparameters": hyperparameters,
        "notes": notes,
        "log_history": log_history or [],
    }
    with open(os.path.join(out_dir, f"run_{tag}.json"), "w") as f:
        json.dump(record, f, indent=2, default=str, ensure_ascii=False)
    print(f"[run record] {tag}: BLEU={scores['bleu']:.2f} chrF++={scores['chrfpp']:.2f} "
          f"GM={scores['geo_mean']:.2f} -> {pred_path}")
    return scores

In [ ]:
# >>> SHARED CELL: common/env.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import importlib
import json
import os
import platform
import sys

TRACKED_PACKAGES = ["torch", "transformers", "datasets", "accelerate", "peft", "bitsandbytes",
                    "sentencepiece", "sacrebleu", "pandas", "numpy", "scikit-learn"]


def report_environment(notebook, out_dir="results"):
    versions = {}
    for name in TRACKED_PACKAGES:
        try:
            module = importlib.import_module("sklearn" if name == "scikit-learn" else name)
            versions[name] = getattr(module, "__version__", "unknown")
        except Exception:
            versions[name] = None
    env = {"notebook": notebook, "python": sys.version.split()[0], "platform": platform.platform(),
           "packages": versions, "gpu": None, "cuda": None}
    try:
        import torch
        if torch.cuda.is_available():
            env["gpu"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
            env["cuda"] = torch.version.cuda
    except Exception:
        pass
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, f"env_{notebook}.json"), "w") as f:
        json.dump(env, f, indent=2)
    print(json.dumps(env, indent=2))
    return env

In [ ]:
report_environment("mbart50")  # -> results/env_mbart50.json

## Data — raw `train.csv`, untouched

No normalization, no sentence-boundary extraction. Only rows with a missing source or target are dropped.

The **document-level** train/validation split comes from the shared `make_split` cell above, so mBART, ByT5, Qwen and the ensemble are all validated on exactly the same 156 documents.

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

RAW_TRAIN_PATH = "/kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv"

# Shared document-level split (identical in every notebook, verified by checksum).
train_df, val_df = make_split(pd.read_csv(RAW_TRAIN_PATH))

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
})

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"
# Local/attached-dataset path used in offline mode (no internet download).
MODEL_PATH = "/kaggle/input/models/nensipansuriya1311/facebookmbart-large-50-many-to-many-mmt/pytorch/default/1"

MODEL_SOURCE = MODEL_NAME if IS_ONLINE else MODEL_PATH

tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE)

tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "en_XX"

# Length limits (Phase 2): chosen per tokenizer from the 95th percentile, so >= 95% of training documents fit
# without truncation (see README, 'Length limits'). Before Phase 2 they were 256.
MAX_SOURCE_LENGTH = 576  # mBART tokens: p95 of transliterations = 568
MAX_TARGET_LENGTH = 448  # mBART tokens: p95 of translations = 413

def preprocess_function(examples):
    inputs = examples["transliteration"]
    targets = examples["translation"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=False
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_SOURCE)

## Metrics

`corpus_scores` from the shared metrics cell: BLEU and chrF++ (character 6-grams **and** word 2-grams), identical to sacreBLEU and computed without internet. During training it is used only to pick the best checkpoint (by the geometric mean); the reported number is computed after training against the full, untruncated references (`finalize_run` below).

In [ ]:
import numpy as np


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace -100 in labels so they decode correctly
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = [p.strip() for p in tokenizer.batch_decode(preds, skip_special_tokens=True)]
    decoded_labels = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]

    s = corpus_scores(decoded_preds, decoded_labels)  # shared metric (sacreBLEU BLEU + chrF++)
    return {"bleu": round(s["bleu"], 4), "chrfpp": round(s["chrfpp"], 4), "geo_mean": round(s["geo_mean"], 4)}

## Training

Memory optimizations added on top of the original setup (all safe to use in both modes):

- **No length grouping** — training batches are sampled randomly. `group_by_length` was removed in transformers 5.2, and `sortish_sampler=True` (kept as in the recorded run) has no effect in `Seq2SeqTrainer`. Padding is dynamic (per batch).
- **`eval_accumulation_steps=1`** — with `predict_with_generate=True`, the Trainer otherwise keeps *all* generated eval predictions resident on the GPU until the full eval pass finishes; this was the most likely source of eval-time OOM, independent of `MAX_TARGET_LENGTH`. Setting this moves each batch's predictions to CPU immediately.
- **Explicit `use_cache` toggling** — `use_cache=True` (the default, used by `.generate()`) is incompatible with `gradient_checkpointing=True` and silently wastes memory/emits warnings during training if left on. It's turned off before `trainer.train()` and back on before generation/prediction.

**Per-epoch MLflow logging** (online mode only, via the `MLflowEpochLogger` callback): `train_loss`, `val_loss`, `chrf_pp`, `bleu`, and `geo_mean_chrf_bleu` (the geometric mean of chrF++ and BLEU) are logged once per epoch, keyed by epoch number, so the run's metric charts in MLflow/DagsHub show all five curves across training. Offline mode skips this entirely — the callback list is empty and training/eval still run and print `eval_metrics` normally.

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, TrainerCallback
# Kaggle Secrets are read only in online mode (offline runs have no access to them).
HF_REPO_NAME = None
if IS_ONLINE:
    from kaggle_secrets import UserSecretsClient
    HF_REPO_NAME = UserSecretsClient().get_secret("HF_REPO_NAME")
RUN_NAME = f"mbart50-train-csv-seed{SEED}"

set_seed(SEED)

training_args = Seq2SeqTrainingArguments(
    output_dir="./results_mbart",
    seed=SEED,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=2,   # effective batch = 2 x 4 x n_gpu (8 on one GPU, 16 on T4 x2)
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="adafactor",
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=1,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="geo_mean",  # competition score
    greater_is_better=True,
    report_to="none",
    disable_tqdm=False,
    sortish_sampler=True,          # kept as in the recorded run; Seq2SeqTrainer does not use it (random sampling)
    eval_accumulation_steps=1,     # stream eval predictions to CPU instead of accumulating on GPU
)

model.gradient_checkpointing_enable()
model.config.use_cache = False  # required alongside gradient checkpointing during training

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


class MLflowEpochLogger(TrainerCallback):
    """
    Logs train loss, val loss, chrF++, BLEU, and their geometric mean to MLflow once
    per epoch (a no-op when IS_ONLINE is False or no MLflow run is active). Training
    loss is captured from the last regular `on_log` call (Trainer logs it every
    `logging_steps`) and paired up with that epoch's eval metrics in `on_evaluate`,
    since eval_strategy="epoch" fires `on_evaluate` once at the end of each epoch.
    """

    def __init__(self):
        self._last_train_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs and "eval_loss" not in logs:
            self._last_train_loss = logs["loss"]

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not IS_ONLINE or metrics is None or mlflow.active_run() is None:
            return

        epoch = int(round(state.epoch)) if state.epoch is not None else state.global_step
        eval_loss = metrics.get("eval_loss")
        chrf_pp = metrics.get("eval_chrfpp")
        bleu = metrics.get("eval_bleu")
        geo_mean = metrics.get("eval_geo_mean")

        epoch_metrics = {}
        if self._last_train_loss is not None:
            epoch_metrics["train_loss"] = self._last_train_loss
        if eval_loss is not None:
            epoch_metrics["val_loss"] = eval_loss
        if chrf_pp is not None:
            epoch_metrics["chrf_pp"] = chrf_pp
        if bleu is not None:
            epoch_metrics["bleu"] = bleu
        if geo_mean is not None:
            epoch_metrics["geo_mean_chrf_bleu"] = geo_mean

        if epoch_metrics:
            mlflow.log_metrics(epoch_metrics, step=epoch)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[MLflowEpochLogger()] if IS_ONLINE else [],
)

if IS_ONLINE:
    with mlflow.start_run(run_name=RUN_NAME):
        mlflow.set_tag("author", "author")
        mlflow.set_tag("model_architecture", "mbart50")
        mlflow.set_tag("hf_repo", HF_REPO_NAME)
        mlflow.set_tag("preprocessing", "none (raw train.csv)")

        mlflow.log_params({
            "model_name": MODEL_SOURCE,
            "lr": training_args.learning_rate,
            "batch_size": training_args.per_device_train_batch_size,
            "epochs": training_args.num_train_epochs,
            "seed": SEED,
            "max_src_len": MAX_SOURCE_LENGTH,
            "max_tgt_len": MAX_TARGET_LENGTH,
            "data_columns": "raw_train_csv_untouched"
        })

        trainer.train()

        eval_metrics = trainer.evaluate()
        mlflow.log_metrics({
            "val_chrF_pp": eval_metrics["eval_chrfpp"],
            "val_bleu": eval_metrics["eval_bleu"],
            "val_loss": eval_metrics["eval_loss"],
            "val_geo_mean_chrf_bleu": eval_metrics["eval_geo_mean"],
        })
else:
    trainer.train()
    eval_metrics = trainer.evaluate()
    print(eval_metrics)

# Re-enable the KV cache now that training/eval (which used gradient checkpointing) is done —
# needed for efficient generation below.
model.config.use_cache = True

In [ ]:
# Model and tokenizer go into ONE directory, so it can be attached directly in ensemble-pipeline.ipynb.
MBART_OUT_DIR = "./mbart50-model"
model.save_pretrained(MBART_OUT_DIR)
tokenizer.save_pretrained(MBART_OUT_DIR)
print(f"Saved model and tokenizer to {MBART_OUT_DIR}")

if IS_ONLINE:
    # Publish the trained model to the Hugging Face Hub
    model.push_to_hub(HF_REPO_NAME, private=True)
    tokenizer.push_to_hub(HF_REPO_NAME, private=True)

if PUBLISH_MODELS:
    import kagglehub
    handle = f"{KAGGLE_USERNAME}/akkadian-mbart/pytorch/mbart50-seed{SEED}"
    kagglehub.model_upload(handle=handle, local_model_dir=MBART_OUT_DIR,
                           version_notes=f"mBART-50 fine-tuned, seed {SEED}, shared split (Phase 2)",
                           license_name="Apache 2.0")
    print("Uploaded:", handle)

In [ ]:
# Validation predictions of the best checkpoint + run record (results/preds_val_mbart50__seed<SEED>.csv,
# results/run_mbart50__seed<SEED>.json). The score printed here is the one reported in the paper.
raw_predictions = trainer.predict(tokenized_datasets["validation"])
preds = np.where(raw_predictions.predictions != -100, raw_predictions.predictions, tokenizer.pad_token_id)
decoded_preds = [p.strip() for p in tokenizer.batch_decode(preds, skip_special_tokens=True)]

finalize_run(
    system="mbart50",
    seed=SEED,
    ids=val_df["oare_id"],
    refs=val_df["translation"],
    preds=decoded_preds,
    hyperparameters={
        "base_model": MODEL_NAME,
        "learning_rate": training_args.learning_rate,
        "per_device_train_batch_size": training_args.per_device_train_batch_size,
        "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
        "effective_batch_size": training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps * max(1, training_args.n_gpu),
        "num_train_epochs": training_args.num_train_epochs,
        "optim": str(training_args.optim),
        "weight_decay": training_args.weight_decay,
        "max_source_length": MAX_SOURCE_LENGTH,
        "max_target_length": MAX_TARGET_LENGTH,
        "generation_num_beams": training_args.generation_num_beams,
        "metric_for_best_model": training_args.metric_for_best_model,
        "best_model_checkpoint": trainer.state.best_model_checkpoint,
    },
    log_history=trainer.state.log_history,
)

## Test-set inference (submission.csv)

Replaces the original row-by-row generation loop with **length-sorted, dynamically-padded mini-batches**:

- Sorting by tokenized length before batching means each batch pads only to the length of its *longest member*, not to a fixed worst case — this cuts wasted compute/memory versus batching in file order.
- Batches (not single examples) get better GPU utilization than the original loop, without the memory risk of one big fixed-size batch, since batches are still capped at `GEN_BATCH_SIZE` and periodically cleared.
- If a batch still OOMs (e.g. a batch of unusually long examples), the batch size is halved and retried automatically instead of crashing the whole run.
- Results are reordered back to the original file order before writing `submission.csv`.

In [ ]:
import os
import gc
import pandas as pd
import torch
from tqdm import tqdm

test_path = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
test_df = pd.read_csv(test_path)

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

forced_bos_token_id = tokenizer.convert_tokens_to_ids("en_XX")  # works across transformers versions

GEN_BATCH_SIZE = 8  # starting batch size

texts = test_df["transliteration"].tolist()
# Sort by tokenized length so each batch pads to its own max, not a global worst case
order = sorted(range(len(texts)), key=lambda i: len(tokenizer(texts[i], truncation=True, max_length=MAX_SOURCE_LENGTH)["input_ids"]))
sorted_texts = [texts[i] for i in order]

predictions_sorted = [None] * len(sorted_texts)

def generate_batch(batch_texts):
    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        padding=True,
    ).to(device)
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            pad_token_id=tokenizer.pad_token_id,
            forced_bos_token_id=forced_bos_token_id,
        )
    return [tokenizer.decode(o, skip_special_tokens=True).strip() for o in outputs]

pbar = tqdm(total=len(sorted_texts), desc="Generating Submissions")
i = 0
batch_size = GEN_BATCH_SIZE
while i < len(sorted_texts):
    chunk = sorted_texts[i:i + batch_size]
    try:
        decoded = generate_batch(chunk)
        for j, text in enumerate(decoded):
            predictions_sorted[i + j] = text
        i += len(chunk)
        pbar.update(len(chunk))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()
        if batch_size == 1:
            raise
        batch_size = max(1, batch_size // 2)
        print(f"CUDA OOM — reducing generation batch size to {batch_size} and retrying.")
    finally:
        if device == "cuda" and i % (GEN_BATCH_SIZE * 4) == 0:
            torch.cuda.empty_cache()
pbar.close()

predictions = [None] * len(texts)
for sorted_idx, original_idx in enumerate(order):
    predictions[original_idx] = predictions_sorted[sorted_idx]

submission = pd.DataFrame({
    "id": test_df["id"],
    "translation": predictions
})

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv successfully!")